In [4]:
# @title Criar Dataset

import google.generativeai as genai
import json
import time
import os
from pathlib import Path
from google.colab import userdata
import requests
from urllib.parse import unquote

# Caminho Local (Raiz do Colab)
LOCAL_DIR = "."

# Lista de URLs Absolutas (Fornecidas Manualmente)
IMAGES_URLS = [
    "https://raw.githubusercontent.com/ErickMBarreto/HackatonFiap/refs/heads/main/dataset/images/aws_1.png",
    "https://raw.githubusercontent.com/ErickMBarreto/HackatonFiap/refs/heads/main/dataset/images/aws_2.png",
    "https://raw.githubusercontent.com/ErickMBarreto/HackatonFiap/refs/heads/main/dataset/images/aws_3.png",
    "https://raw.githubusercontent.com/ErickMBarreto/HackatonFiap/refs/heads/main/dataset/images/azure_1.png",
    "https://raw.githubusercontent.com/ErickMBarreto/HackatonFiap/refs/heads/main/dataset/images/azure_2.png",
    "https://raw.githubusercontent.com/ErickMBarreto/HackatonFiap/refs/heads/main/dataset/images/azure_3.png"

]

def download_source_images():
    print("INICIALIZANDO DOWNLOADER DE IMAGENS FONTE (MODO URL ABSOLUTA)...")
    print(f"Baixando {len(IMAGES_URLS)} imagens para a raiz do ambiente...")

    for remote_url in IMAGES_URLS:
        filename = unquote(remote_url.split('/')[-1])
        local_path = os.path.join(LOCAL_DIR, filename)

        if not os.path.exists(local_path):
            try:
                response = requests.get(remote_url)
                if response.status_code == 200:
                    with open(local_path, 'wb') as f:
                        f.write(response.content)
                    print(f"  [200 OK] Baixado: {filename}")
                else:
                    print(f"  [ERRO {response.status_code}] Falha ao baixar {filename}")
            except Exception as e:
                print(f"  Falha de conexão: {e}")
        else:
            print(f"  Arquivo já existe: {filename}")

    print(" Download concluído. As imagens estão prontas para o Bootstrapper.\n")

# Executar
download_source_images()

# --- CONFIGURAÇÃO ---
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except:
    API_KEY = input("Cole sua API Key novamente: ")

genai.configure(api_key=API_KEY)
MODEL_NAME = "models/gemini-2.5-flash" # Mantemos o professor inteligente

generation_config = {
  "temperature": 0.1,
  "top_p": 0.95,
  "top_k": 64,
  "max_output_tokens": 8192,
  "response_mime_type": "application/json", # Força modo JSON
}

# PROMPT AJUSTADO PARA EVITAR ESTOURO DE TOKENS
model = genai.GenerativeModel(
  model_name=MODEL_NAME,
  generation_config=generation_config,
  system_instruction="""
  ATENÇÃO: CRITICAL JSON SYNTAX RULE.
  Você deve gerar um JSON válido.
  - Não deixe strings abertas.
  - Escape aspas internas com \".
  - Não adicione comentários // no JSON.

  Tarefa: Analise o diagrama e gere o relatório STRIDE.

  CRÍTICO: SEJA CONCISO.
  Não escreva descrições longas. Use frases curtas e diretas para Ameaças e Mitigações.
  Isso é essencial para não estourar o limite de tokens e quebrar o JSON.

  Estrutura:
  {
    "architecture_title": "string",
    "components": [
       { "id": "str", "name": "str", "type": "str", "stride_analysis": [{"category": "str", "threat": "str (max 20 words)", "mitigation": "str (max 20 words)"}] }
    ]
  }
  """
)

# --- LÓGICA DE REPESCAGEM ---

# Verifica se o usuário fez o upload do arquivo anterior
if not os.path.exists('dataset_treino.json'):
    print(" AVISO: Não encontrei o arquivo 'dataset_treino.json'.")
    print("Se você quiser aproveitar o que já foi gerado, faça o upload dele agora.")
    print("Caso contrário, o script vai processar TODAS as imagens do zero.")
    time.sleep(5) # Dá um tempo para ler o aviso

# 1. Carrega o que já deu certo
try:
    with open('dataset_treino.json', 'r', encoding='utf-8') as f:
        sucesso_anterior = json.load(f)
    print(f" Carregados {len(sucesso_anterior)} itens já processados com sucesso.")
    imagens_processadas = [item['image_filename'] for item in sucesso_anterior]
except FileNotFoundError:
    print(" Começando dataset do zero (modo full generation).")
    sucesso_anterior = []
    imagens_processadas = []

# 2. Identifica o que falta
image_extensions = ['*.png', '*.jpg', '*.jpeg']
todas_imagens = []
for ext in image_extensions:
    todas_imagens.extend(Path('.').glob(ext))

imagens_para_processar = [img for img in todas_imagens if img.name not in imagens_processadas]

if not imagens_para_processar:
    print(" Nada a processar! Todas as imagens já estão no JSON.")
else:
    print(f" Iniciando REPESCAGEM para {len(imagens_para_processar)} imagens...\n")

    novos_itens = []

    for img_path in imagens_para_processar:
        print(f"   > Tentando novamente: {img_path.name}...")
        try:
            sample_file = genai.upload_file(path=img_path, display_name=img_path.name)

            while sample_file.state.name == "PROCESSING":
                time.sleep(1)
                sample_file = genai.get_file(sample_file.name)

            # Prompt reforçado para brevidade
            response = model.generate_content([sample_file, "Gere o JSON STRIDE. Seja conciso e breve para garantir JSON válido."])

            json_content = json.loads(response.text)

            novos_itens.append({
                "image_filename": img_path.name,
                "ground_truth_json": json_content
            })
            print(f"      SUCESSO! Recuperada.")

        except Exception as e:
            print(f"      Falhou de novo: {e}")

    # 3. Salva o arquivo FINAL MESCLADO
    lista_final = sucesso_anterior + novos_itens
    with open('dataset_treino.json', 'w', encoding='utf-8') as f:
        json.dump(lista_final, f, indent=2, ensure_ascii=False)

    print(f"\n PROCESSO FINALIZADO!!.")
    print(f"Total de itens no dataset: {len(lista_final)} de {len(todas_imagens)}")
    print("Baixe o arquivo 'dataset_treino.json'")

INICIALIZANDO DOWNLOADER DE IMAGENS FONTE (MODO URL ABSOLUTA)...
Baixando 6 imagens para a raiz do ambiente...
  Arquivo já existe: aws_1.png
  Arquivo já existe: aws_2.png
  Arquivo já existe: aws_3.png
  Arquivo já existe: azure_1.png
  Arquivo já existe: azure_2.png
  Arquivo já existe: azure_3.png
 Download concluído. As imagens estão prontas para o Bootstrapper.

 Carregados 5 itens já processados com sucesso.
 Iniciando REPESCAGEM para 1 imagens...

   > Tentando novamente: azure_2.png...
      SUCESSO! Recuperada.

 PROCESSO FINALIZADO!!.
Total de itens no dataset: 6 de 6
Baixe o arquivo 'dataset_treino.json'
